In [0]:
import mlflow
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

catalog = "workspace"
schema = "financial_sentiment"
table_prefix = f"{catalog}.{schema}"

model_name = f"{catalog}.{schema}.financial_sentiment_classifier"

mlflow.set_registry_uri("databricks-uc")

model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

model = mlflow.sklearn.load_model(model_uri)

texts = pd.DataFrame({
    "text_clean": [
        "the company reported strong revenue growth and better margins",
        "the bank faces losses due to market uncertainty",
        "the quarterly report was in line with expectations"
    ]
})

texts["prediction"] = model.predict(texts)
texts["model_name"] = model_name
texts["model_version"] = model_version
texts["prediction_timestamp"] = datetime.now()

display(texts)

pred_spark = spark.createDataFrame(texts)

pred_spark.write.mode("append").saveAsTable(
    f"{table_prefix}.gold_financial_sentiment_predictions"
)

display(
    spark.table(f"{table_prefix}.gold_financial_sentiment_predictions")
)